### 1. Import Libraries & Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# MỤC TIÊU: Huấn luyện mô hình GRU (Gated Recurrent Unit)
# để dự báo doanh số bán hàng, sử dụng dữ liệu đã qua xử lý (feature engineered data).
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.nn.utils import weight_norm
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

#### 2. Load Data

In [4]:
data_train_set = pd.read_parquet('drive/MyDrive/data/model/train_fe.parquet')
data_test_set = pd.read_parquet('drive/MyDrive/data/model/test_fe.parquet')

In [5]:
data_train_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830972 entries, 0 to 830971
Data columns (total 28 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Store                    830972 non-null  int64  
 1   DayOfWeek                830972 non-null  int32  
 2   Sales                    830972 non-null  float64
 3   Customers                830972 non-null  int64  
 4   Open                     830972 non-null  int64  
 5   Promo                    830972 non-null  int64  
 6   StateHoliday             830972 non-null  object 
 7   SchoolHoliday            830972 non-null  int64  
 8   StoreType                830972 non-null  object 
 9   Assortment               830972 non-null  object 
 10  CompetitionDistance      830972 non-null  float64
 11  Promo2                   830972 non-null  int64  
 12  CompetitionMissingFlag   830972 non-null  int64  
 13  LogSales                 830972 non-null  float64
 14  Year

In [6]:
data_test_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1113 entries, 0 to 1112
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Store                    1113 non-null   int64  
 1   DayOfWeek                1113 non-null   int32  
 2   Open                     1113 non-null   int64  
 3   Promo                    1113 non-null   int64  
 4   StateHoliday             1113 non-null   object 
 5   SchoolHoliday            1113 non-null   int64  
 6   StoreType                1113 non-null   object 
 7   Assortment               1113 non-null   object 
 8   CompetitionDistance      1113 non-null   float64
 9   Promo2                   1113 non-null   int64  
 10  CompetitionMissingFlag   1113 non-null   int64  
 11  LogSales                 0 non-null      float64
 12  Year                     1113 non-null   int32  
 13  Month                    1113 non-null   int32  
 14  Day                     

### 3. Data Preprocessing & Feature Engineering

In [7]:
TARGET_FIELD = "Sales"
STORE_ID_FIELD = "Store"
DATE_COMPONENTS = ["Year", "Month", "Day"]
CATEGORY_FIELDS = ["StateHoliday", "StoreType", "Assortment"]
EXCLUDED_FIELDS = {TARGET_FIELD, "Customers", "LogSales", "Date"}
MISSING_TO_ZERO_FIELDS = [
    "CompetitionMonthsActive",
    "Promo2WeeksActive",
    "Lag_1",
    "Lag_7",
    "Rolling_Mean_7",
    "Rolling_Std_7",
]

# Chuẩn bị cột mục tiêu cho tập kiểm tra (nếu chưa có)
for col_name in [TARGET_FIELD, "Customers", "LogSales"]:
    if col_name not in data_test_set.columns:
        data_test_set[col_name] = np.nan

# Xây dựng cột Date từ các thành phần Year/Month/Day
for dataframe in [data_train_set, data_test_set]:
    date_info = dataframe[DATE_COMPONENTS].rename(columns={"Year": "year", "Month": "month", "Day": "day"})
    dataframe["Date"] = pd.to_datetime(date_info)

# Kết hợp hai tập dữ liệu để xử lý đồng bộ
data_train_set["dataset_type"] = "train"
data_test_set["dataset_type"] = "test"
combined_data = pd.concat([data_train_set, data_test_set], ignore_index=True)

# Chuẩn hóa các trường phân loại (categorical)
for col_cat in CATEGORY_FIELDS:
    combined_data[col_cat] = combined_data[col_cat].astype(str)

# Điền giá trị NA bằng 0 cho các cột quy định
for col_na in MISSING_TO_ZERO_FIELDS:
    if col_na in combined_data.columns:
        combined_data[col_na] = combined_data[col_na].fillna(0)

# Thực hiện One-hot encoding
combined_data = pd.get_dummies(combined_data, columns=CATEGORY_FIELDS, drop_first=True)
combined_data = combined_data.sort_values([STORE_ID_FIELD, "Date"]).reset_index(drop=True)

# Tách lại Train và Test
data_train_set = combined_data[combined_data["dataset_type"] == "train"].drop(columns=["dataset_type"]).reset_index(drop=True)
data_test_set = combined_data[combined_data["dataset_type"] == "test"].drop(columns=["dataset_type"]).reset_index(drop=True)

MODEL_FEATURES = [col for col in data_train_set.columns if col not in EXCLUDED_FIELDS and col != STORE_ID_FIELD]

print(f"Kích thước tập huấn luyện (sau encoding): {data_train_set.shape}")
print(f"Kích thước tập kiểm tra (sau encoding): {data_test_set.shape}")
print(f"Tổng số features: {len(MODEL_FEATURES)}")

Kích thước tập huấn luyện (sau encoding): (830972, 34)
Kích thước tập kiểm tra (sau encoding): (1113, 34)
Tổng số features: 29


### 4. Train/Validation Split & Scaling

In [8]:
VALIDATION_DURATION_WEEKS = 6
validation_period = pd.Timedelta(weeks=VALIDATION_DURATION_WEEKS)
split_point = data_train_set["Date"].max() - validation_period

data_train_core = data_train_set[data_train_set["Date"] < split_point].copy()
data_val_core = data_train_set[data_train_set["Date"] >= split_point].copy()

# Khởi tạo và fit các bộ scaler trên tập huấn luyện
feature_normalizer = StandardScaler()
target_normalizer = StandardScaler()

# Chuẩn hóa Features (X)
data_train_core[MODEL_FEATURES] = feature_normalizer.fit_transform(data_train_core[MODEL_FEATURES])
data_val_core[MODEL_FEATURES] = feature_normalizer.transform(data_val_core[MODEL_FEATURES])
data_test_normalized = data_test_set.copy()
data_test_normalized[MODEL_FEATURES] = feature_normalizer.transform(data_test_normalized[MODEL_FEATURES])

# Chuẩn hóa Target (Y)
data_train_core[[TARGET_FIELD]] = target_normalizer.fit_transform(data_train_core[[TARGET_FIELD]])
data_val_core[[TARGET_FIELD]] = target_normalizer.transform(data_val_core[[TARGET_FIELD]])

print(f"Ngày chia tập dữ liệu: {split_point.date()}")
print(f"Số mẫu huấn luyện: {len(data_train_core):,} | Số mẫu kiểm định: {len(data_val_core):,}")

Ngày chia tập dữ liệu: 2015-06-05
Số mẫu huấn luyện: 789,557 | Số mẫu kiểm định: 41,415


### 5. Sequence Building for GRU

In [9]:
SEQUENCE_LENGTH = 30  # Số ngày lịch sử để dự đoán
BATCH_SIZE_VAL = 256

def generate_sequences(df_input: pd.DataFrame, feature_list, target_field_name):
    """Hàm tạo các chuỗi (sequences) input và target cho mô hình học sâu"""
    input_sequences, output_targets = [], []
    for _, group_data in df_input.groupby(STORE_ID_FIELD):
        group_data = group_data.sort_values("Date")
        matrix_values = group_data[feature_list + [target_field_name]].to_numpy()
        if len(matrix_values) <= SEQUENCE_LENGTH:
            continue
        for idx_start in range(len(matrix_values) - SEQUENCE_LENGTH):
            seq_input_x = matrix_values[idx_start:idx_start + SEQUENCE_LENGTH, :-1]
            seq_target_y = matrix_values[idx_start + SEQUENCE_LENGTH, -1]
            input_sequences.append(seq_input_x)
            output_targets.append(seq_target_y)
    return np.array(input_sequences, dtype=np.float32), np.array(output_targets, dtype=np.float32)

X_seq_train, y_seq_train = generate_sequences(data_train_core, MODEL_FEATURES, TARGET_FIELD)
X_seq_val, y_seq_val = generate_sequences(data_val_core, MODEL_FEATURES, TARGET_FIELD)

print(f"Kích thước chuỗi Train: {X_seq_train.shape}")
print(f"Kích thước chuỗi Val  : {X_seq_val.shape}")

Kích thước chuỗi Train: (756107, 30, 29)
Kích thước chuỗi Val  : (7977, 30, 29)


### 6. Dataset & DataLoader Setup

In [10]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X_data, y_data=None):
        self.X_tensor = torch.tensor(X_data, dtype=torch.float32)
        self.y_tensor = torch.tensor(y_data, dtype=torch.float32) if y_data is not None else None

    def __len__(self):
        return len(self.X_tensor)

    def __getitem__(self, idx):
        if self.y_tensor is None:
            return self.X_tensor[idx]
        return self.X_tensor[idx], self.y_tensor[idx]

train_data_set = TimeSeriesDataset(X_seq_train, y_seq_train)
val_data_set = TimeSeriesDataset(X_seq_val, y_seq_val)

train_data_loader = DataLoader(train_data_set, batch_size=BATCH_SIZE_VAL, shuffle=True, drop_last=False)
val_data_loader = DataLoader(val_data_set, batch_size=BATCH_SIZE_VAL, shuffle=False, drop_last=False)

print(f"Số lượng batches -> huấn luyện: {len(train_data_loader)}, kiểm định: {len(val_data_loader)}")

Số lượng batches -> huấn luyện: 2954, kiểm định: 32


### 7. GRU Architecture Definition

In [11]:
class SalesForecasterGRU(nn.Module):
    """Mô hình GRU để dự báo doanh số bán hàng"""
    def __init__(self, input_size, hidden_size, layer_count=2, drop_rate=0.2):
        super(SalesForecasterGRU, self).__init__()

        # Định nghĩa lớp GRU
        self.gru_layer = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=layer_count,
            batch_first=True,
            dropout=drop_rate
        )

        # Lớp đầu ra (Fully connected)
        self.output_linear = nn.Linear(hidden_size, 1)
        self.drop_layer = nn.Dropout(drop_rate)

    def forward(self, input_sequence):
        # input_sequence shape: (batch_size, sequence_length, input_size)

        # Truyền qua GRU
        # output: (batch_size, sequence_length, hidden_size)
        # h_n: (layer_count, batch_size, hidden_size)
        gru_output, final_hidden = self.gru_layer(input_sequence)

        # Lấy hidden state của *phần tử cuối cùng* trong chuỗi
        # last_timestep_output shape: (batch_size, hidden_size)
        last_timestep_output = gru_output[:, -1, :]

        # Lớp tuyến tính để dự đoán giá trị (univariate forecast)
        final_prediction = self.output_linear(self.drop_layer(last_timestep_output))
        return final_prediction.squeeze(-1)

### 8. Model Configuration & Initialization

In [12]:
INPUT_FEATURE_COUNT = len(MODEL_FEATURES)
GRU_HIDDEN_SIZE = 128
GRU_LAYER_COUNT = 2
DROPOUT_RATE = 0.2
EXECUTION_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Khởi tạo mô hình GRU
forecasting_model = SalesForecasterGRU(INPUT_FEATURE_COUNT, GRU_HIDDEN_SIZE, GRU_LAYER_COUNT, DROPOUT_RATE).to(EXECUTION_DEVICE)

print(f"\nThiết bị chạy: {EXECUTION_DEVICE}")
print(f"\nKiến trúc Mô hình:")
print(forecasting_model)

# Đếm số parameters
total_parameters = sum(p.numel() for p in forecasting_model.parameters())
trainable_parameters = sum(p.numel() for p in forecasting_model.parameters() if p.requires_grad)
print(f"\nTổng số tham số: {total_parameters:,}")
print(f"Tham số có thể huấn luyện: {trainable_parameters:,}")


Thiết bị chạy: cuda

Kiến trúc Mô hình:
SalesForecasterGRU(
  (gru_layer): GRU(29, 128, num_layers=2, batch_first=True, dropout=0.2)
  (output_linear): Linear(in_features=128, out_features=1, bias=True)
  (drop_layer): Dropout(p=0.2, inplace=False)
)

Tổng số tham số: 160,257
Tham số có thể huấn luyện: 160,257


### 9. Training Setup

In [13]:
MAX_EPOCHS = 10
INITIAL_LEARNING_RATE = 1e-3
EARLY_STOPPING_PATIENCE = 5

loss_fn = nn.MSELoss()
optimizer_fn = torch.optim.AdamW(forecasting_model.parameters(), lr=INITIAL_LEARNING_RATE, weight_decay=1e-5)

# Định nghĩa Learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fn, mode='min', factor=0.5, patience=3
)

def execute_one_epoch(data_loader, is_training=True):
    """Thực hiện một chu trình huấn luyện hoặc đánh giá"""
    running_loss, running_mae = 0.0, 0.0
    iteration_steps = 0

    if is_training:
        forecasting_model.train()
    else:
        forecasting_model.eval()

    for batch_data in data_loader:
        features_batch, targets_batch = [b.to(EXECUTION_DEVICE) for b in batch_data]

        if is_training:
            optimizer_fn.zero_grad()

        with torch.set_grad_enabled(is_training):
            predictions = forecasting_model(features_batch)
            loss_value = loss_fn(predictions, targets_batch)
            mae_value = torch.mean(torch.abs(predictions - targets_batch))

            if is_training:
                loss_value.backward()
                torch.nn.utils.clip_grad_norm_(forecasting_model.parameters(), max_norm=1.0)
                optimizer_fn.step()

        running_loss += loss_value.item()
        running_mae += mae_value.item()
        iteration_steps += 1

    return running_loss / iteration_steps, running_mae / iteration_steps

### 10. Training Loop

In [22]:
print("KHỞI CHẠY HUẤN LUYỆN MÔ HÌNH GRU")


best_validation_loss = float("inf")
patience_counter_current = 0
training_log = []

for epoch_num in range(1, MAX_EPOCHS + 1):
    train_mse, train_mae = execute_one_epoch(train_data_loader, is_training=True)
    val_mse, val_mae = execute_one_epoch(val_data_loader, is_training=False)

    training_log.append({
        "epoch": epoch_num,
        "train_loss": train_mse,
        "val_loss": val_mse,
        "val_mae": val_mae
    })

    print(f"Epoch {epoch_num:02d} | Train Loss (MSE) {train_mse:.4f} | Val Loss (MSE) {val_mse:.4f} | Val MAE {val_mae:.4f}")

    # Điều chỉnh Learning rate
    lr_scheduler.step(val_mse)

    # Dừng sớm & lưu checkpoint
    if val_mse < best_validation_loss:
        best_validation_loss = val_mse
        patience_counter_current = 0
        torch.save(forecasting_model.state_dict(), "gru_best.pt")
        print(f"  → Đã lưu mô hình tốt nhất (val_loss: {val_mse:.4f})")
    else:
        patience_counter_current += 1
        if patience_counter_current >= EARLY_STOPPING_PATIENCE:
            print(f"\nDừng sớm được kích hoạt sau {epoch_num} epochs")
            break

print("QUÁ TRÌNH HUẤN LUYỆN ĐÃ HOÀN TẤT")

KHỞI CHẠY HUẤN LUYỆN MÔ HÌNH GRU
Epoch 01 | Train Loss (MSE) 0.0487 | Val Loss (MSE) 0.1068 | Val MAE 0.2300
  → Đã lưu mô hình tốt nhất (val_loss: 0.1068)
Epoch 02 | Train Loss (MSE) 0.0482 | Val Loss (MSE) 0.1202 | Val MAE 0.2417
Epoch 03 | Train Loss (MSE) 0.0479 | Val Loss (MSE) 0.1214 | Val MAE 0.2392
Epoch 04 | Train Loss (MSE) 0.0475 | Val Loss (MSE) 0.1238 | Val MAE 0.2438
Epoch 05 | Train Loss (MSE) 0.0461 | Val Loss (MSE) 0.1118 | Val MAE 0.2323
Epoch 06 | Train Loss (MSE) 0.0460 | Val Loss (MSE) 0.1185 | Val MAE 0.2373

Dừng sớm được kích hoạt sau 6 epochs
QUÁ TRÌNH HUẤN LUYỆN ĐÃ HOÀN TẤT


### 11. Evaluation Metrics

In [23]:
def calculate_mape(y_actual, y_predicted):
    """Tính toán Lỗi Phần trăm Tuyệt đối Trung bình (MAPE)"""
    non_zero_elements = y_actual != 0
    if np.sum(non_zero_elements) == 0:
        return np.nan
    y_actual_filtered = y_actual[non_zero_elements]
    y_predicted_filtered = y_predicted[non_zero_elements]
    return np.mean(np.abs((y_actual_filtered - y_predicted_filtered) / y_actual_filtered)) * 100

def calculate_rmspe(y_actual, y_predicted):
    """Tính toán Căn bậc hai Lỗi Phần trăm Bình phương Trung bình (RMSPE)"""
    non_zero_elements = y_actual != 0
    if np.sum(non_zero_elements) == 0:
        return np.nan
    y_actual_filtered = y_actual[non_zero_elements]
    y_predicted_filtered = y_predicted[non_zero_elements]
    return np.sqrt(np.mean(((y_actual_filtered - y_predicted_filtered) / y_actual_filtered)**2)) * 100

def calculate_smape(y_actual, y_predicted):
    """Tính toán Lỗi Phần trăm Tuyệt đối Trung bình Đối xứng (sMAPE)"""
    denominator = (np.abs(y_actual) + np.abs(y_predicted)) / 2
    ratio_safe = np.where(denominator == 0, 0, np.abs(y_predicted - y_actual) / denominator)
    return np.mean(ratio_safe) * 100

def calculate_mase(y_actual, y_predicted, error_baseline_naive):
    """Tính toán Lỗi Tuyệt đối Trung bình được Chia tỷ lệ (MASE)"""
    if np.isnan(error_baseline_naive) or error_baseline_naive == 0:
        return np.nan
    mae_current = mean_absolute_error(y_actual, y_predicted)
    return mae_current / error_baseline_naive

### 12. Final Validation Evaluation

In [24]:
print("ĐÁNH GIÁ TRÊN TẬP KIỂM ĐỊNH")

# Tải trọng số tốt nhất
best_model_path = "gru_best.pt"
if os.path.exists(best_model_path):
    forecasting_model.load_state_dict(torch.load(best_model_path, map_location=EXECUTION_DEVICE))
    print("✓ Đã tải trọng số mô hình tốt nhất")

forecasting_model.eval()

# Tạo dự đoán
all_predictions_scaled, all_targets_scaled = [], []
with torch.no_grad():
    for features_batch, targets_batch in val_data_loader:
        features_batch = features_batch.to(EXECUTION_DEVICE)
        preds_batch = forecasting_model(features_batch).cpu().numpy()
        all_predictions_scaled.append(preds_batch)
        all_targets_scaled.append(targets_batch.numpy())

val_predictions_scaled = np.concatenate(all_predictions_scaled)
val_targets_array_scaled = np.concatenate(all_targets_scaled)

# Đảo ngược chuẩn hóa về thang đo gốc
val_predictions_raw = target_normalizer.inverse_transform(val_predictions_scaled.reshape(-1, 1)).ravel()
val_targets_raw = target_normalizer.inverse_transform(val_targets_array_scaled.reshape(-1, 1)).ravel()

# Tính toán các chỉ số
mae_result = mean_absolute_error(val_targets_raw, val_predictions_raw)
rmse_result = np.sqrt(mean_squared_error(val_targets_raw, val_predictions_raw))
mape_result = calculate_mape(val_targets_raw, val_predictions_raw)
rmspe_result = calculate_rmspe(val_targets_raw, val_predictions_raw)
smape_result = calculate_smape(val_targets_raw, val_predictions_raw)

# Tính toán MASE
# Tính lỗi dự báo naive trên tập huấn luyện gốc
original_train_sales_data = data_train_set[data_train_set["Date"] < split_point][TARGET_FIELD].values
if len(original_train_sales_data) > 1:
    naive_error_baseline = np.mean(np.abs(original_train_sales_data[1:] - original_train_sales_data[:-1]))
    mase_result = calculate_mase(val_targets_raw, val_predictions_raw, naive_error_baseline)
else:
    mase_result = np.nan

# In kết quả
print(f"\nChỉ số đánh giá:")
print(f"  MAE  : {mae_result:,.2f}")
print(f"  RMSE : {rmse_result:,.2f}")
print(f"  MAPE : {mape_result:,.2f}%")
print(f"  RMSPE: {rmspe_result:,.2f}%")
print(f"  sMAPE: {smape_result:,.2f}%")
print(f"  MASE : {mase_result:,.2f}")

ĐÁNH GIÁ TRÊN TẬP KIỂM ĐỊNH
✓ Đã tải trọng số mô hình tốt nhất

Chỉ số đánh giá:
  MAE  : 706.24
  RMSE : 998.88
  MAPE : 9.49%
  RMSPE: 14.07%
  sMAPE: 9.44%
  MASE : 0.51


### 13. Test Set Inference

In [25]:
print("DỰ ĐOÁN TRÊN TẬP KIỂM TRA (TEST SET)")

def generate_test_sequences(test_normalized_df, historical_normalized_df, feature_list, seq_len):
    """Tạo chuỗi đầu vào cho tập kiểm tra từ dữ liệu lịch sử đã chuẩn hóa"""
    input_sequences, store_ids_list, forecast_dates_list = [], [], []

    for index, current_row in test_normalized_df.iterrows():
        current_store_id = current_row[STORE_ID_FIELD]
        current_forecast_date = current_row["Date"]
        sequence_end_date = current_forecast_date - pd.Timedelta(days=1)

        store_historical_data = historical_normalized_df[
            (historical_normalized_df[STORE_ID_FIELD] == current_store_id) &\
            (historical_normalized_df["Date"] <= sequence_end_date)
        ].sort_values("Date")

        if len(store_historical_data) < seq_len:
            continue

        # Lấy N ngày gần nhất (N = seq_len)
        sequence_features = store_historical_data[feature_list].tail(seq_len).to_numpy()
        input_sequences.append(sequence_features.astype(np.float32))
        store_ids_list.append(current_store_id)
        forecast_dates_list.append(current_forecast_date)

    return np.array(input_sequences), store_ids_list, forecast_dates_list

# Chuẩn bị dữ liệu lịch sử đã chuẩn hóa đầy đủ
full_historical_normalized_df = data_train_set.copy()
full_historical_normalized_df[MODEL_FEATURES] = feature_normalizer.transform(data_train_set[MODEL_FEATURES])
full_historical_normalized_df = full_historical_normalized_df.sort_values([STORE_ID_FIELD, "Date"]).reset_index(drop=True)

# Xây dựng chuỗi test
X_seq_test, test_ids, test_dates = generate_test_sequences(
    data_test_normalized,
    full_historical_normalized_df,
    MODEL_FEATURES,
    SEQUENCE_LENGTH
)
print(f"Kích thước chuỗi Test: {X_seq_test.shape}")

# Tạo DataLoader cho Test
test_data_set_final = TimeSeriesDataset(X_seq_test)
test_data_loader_final = DataLoader(test_data_set_final, batch_size=BATCH_SIZE_VAL, shuffle=False)

# Tạo dự đoán
forecasting_model.eval()
test_predictions_scaled_list = []
with torch.no_grad():
    for batch_features in test_data_loader_final:
        predictions_batch = forecasting_model(batch_features.to(EXECUTION_DEVICE)).cpu().numpy()
        test_predictions_scaled_list.append(predictions_batch)

test_predictions_scaled = np.concatenate(test_predictions_scaled_list)
test_predictions_raw = target_normalizer.inverse_transform(test_predictions_scaled.reshape(-1, 1)).ravel()

# Tạo file submission
submission_output = pd.DataFrame({
    "Store": test_ids,
    "ForecastDate": test_dates,
    "PredictedSales": test_predictions_raw
})

output_file_name = "gru_predictions.csv"
submission_output.to_csv(output_file_name, index=False)
print(f"\n✓ Đã lưu kết quả dự đoán vào: {output_file_name}")

DỰ ĐOÁN TRÊN TẬP KIỂM TRA (TEST SET)
Kích thước chuỗi Test: (1113, 30, 29)

✓ Đã lưu kết quả dự đoán vào: gru_predictions.csv
